# Create Planet.com Orders for After-Event Imagery

**Purpose:** Creates Planet Orders API v2 orders for post-event (after) satellite
imagery of landslide incidents. You specify an index range into the CSV, and this
notebook searches for the best available Planet scene over each incident AOI, then
submits an order clipped to that AOI.

**Workflow:**
1. Reads the landslide incidents CSV
2. Filters to the user-specified index range (`start_idx` .. `end_idx`)
3. For each incident, searches Planet's catalog for post-event imagery
4. Picks the least-cloudy scene and creates an order with a clip-to-AOI tool
5. Order names follow the existing naming convention — `incident_<ID>_after`

**Download separately:** Use `planet_orders_download.ipynb` to download the
fulfilled orders.

**Setup (one time):**
1. Get your Planet API key from https://www.planet.com/account/#/
2. In Kaggle: **Add-ons → Secrets** → add a secret named `planet_api_key`
   with your API key as the value, and attach it to this notebook.

**AOI cropping** follows the same logic as `extracting_data.ipynb`:
clamp to `MAX_AOI_DEG = 0.1` degrees (≈11 km), then create a rectangle.
Only **after** images are ordered.

In [ ]:
# --------------------------------------------------------------------
# Imports
# --------------------------------------------------------------------
import pandas as pd
import requests
from requests.adapters import HTTPAdapter, Retry
from datetime import timedelta
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

In [ ]:
# --------------------------------------------------------------------
# Configuration — edit these before running
# --------------------------------------------------------------------

# CSV containing landslide incidents (exact same source as extracting_data.ipynb)
input_csv = "/kaggle/input/datasets/sanjayashrestha123/landslide-reproted/landslides_from_2018_to_2026.csv"

# Index range into the CSV (df.iloc[start_idx:end_idx])
start_idx = 0
end_idx   = 10   # set to e.g. 50 to process incidents 0..49

# Planet item type and product bundle (customise for your subscription)
PLANET_ITEM_TYPE    = "PSScene"               # PlanetScope 3 m
PLANET_BUNDLE       = "analytic_8b_sr_udm2"  # 8-band surface reflectance + UDM2

# AOI clamping (same as extracting_data.ipynb)
MAX_AOI_DEG = 0.1

# Post-event search window (days after incident)
POST_WINDOW_START = 5     # days after incident
POST_WINDOW_END   = 180   # days after incident

# Max cloud cover fraction for scene selection (0..1)
MAX_CLOUD_FRACTION = 0.2

# Planet Orders API
ORDERS_URL = "https://api.planet.com/compute/ops/orders/v2"
SEARCH_URL = "https://api.planet.com/data/v2/quick-search"

# Concurrency
MAX_WORKERS = 4
REQUEST_TIMEOUT = 120

# HuggingFace repo — check here before ordering
HF_REPO_ID = "sasudo2/landslides"
HF_REVISION = "main"

# --------------------------------------------------------------------
# Authenticate: HuggingFace
# --------------------------------------------------------------------
user_secrets = UserSecretsClient()
hf_key = user_secrets.get_secret("huggingface_token")
hf_api = HfApi(token=hf_key)
hf_api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", exist_ok=True)
print(f"HuggingFace repo '{HF_REPO_ID}' ready.")

# --------------------------------------------------------------------
# Authenticate: Planet
# --------------------------------------------------------------------
PLANET_API_KEY = user_secrets.get_secret("planet_api_key")


def make_session():
    s = requests.Session()
    s.auth = (PLANET_API_KEY, "")
    retries = Retry(
        total=5,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET", "POST"]),
        respect_retry_after_header=True,
    )
    s.mount("https://", HTTPAdapter(max_retries=retries, pool_maxsize=MAX_WORKERS * 2))
    return s


session = make_session()

# Sanity check
r = session.get(ORDERS_URL, timeout=60)
r.raise_for_status()
print("Planet authentication OK.")

In [ ]:
# --------------------------------------------------------------------
# Load the landslide incidents CSV and apply index range
# --------------------------------------------------------------------
df = pd.read_csv(input_csv)
df = df.iloc[start_idx:end_idx]
df['incident_on'] = pd.to_datetime(df['incident_on'], dayfirst=True)
print(f"Loaded {len(df)} incidents (rows {start_idx}:{end_idx}).")

In [ ]:
import re

# --------------------------------------------------------------------
# Check HF repo for existing after images — skip already-downloaded
# --------------------------------------------------------------------
print("Checking HuggingFace repo for existing images...")
existing_after_ids = set()
try:
    for f in hf_api.list_repo_files(HF_REPO_ID, repo_type="dataset"):
        m = re.match(r'^incident_(\d+)/', f)
        if m and f.endswith('_after.tif'):
            existing_after_ids.add(int(m.group(1)))
    print(f"Found {len(existing_after_ids)} incidents with _after.tif in repo.")

    before = len(df)
    df = df[~df['id'].isin(existing_after_ids)]
    skipped = before - len(df)
    if skipped:
        print(f"Skipping {skipped} incidents already in repo.")
    print(f"Processing {len(df)} incidents.")
except Exception as e:
    print(f"Could not query HF repo: {e}")
    print("Proceeding with all incidents.")

## Helper functions

These mirror the AOI logic from `extracting_data.ipynb` (`clamp_aoi`) and add
Planet-specific search and order-creation helpers.

In [ ]:
# --------------------------------------------------------------------
# AOI helper — identical to extracting_data.ipynb
# --------------------------------------------------------------------
def clamp_aoi(min_lon, min_lat, max_lon, max_lat):
    """Clamp AOI extent to MAX_AOI_DEG if either dimension exceeds it.
    Same logic as extracting_data.ipynb.
    """
    lon_span = max_lon - min_lon
    lat_span = max_lat - min_lat
    if lon_span <= MAX_AOI_DEG and lat_span <= MAX_AOI_DEG:
        return min_lon, min_lat, max_lon, max_lat
    cx = (min_lon + max_lon) / 2
    cy = (min_lat + max_lat) / 2
    half = MAX_AOI_DEG / 2
    return cx - half, cy - half, cx + half, cy + half


def aoi_to_planet_geometry(min_lon, min_lat, max_lon, max_lat):
    """Convert clamped AOI to a Planet GeoJSON Polygon."""
    c_min_lon, c_min_lat, c_max_lon, c_max_lat = clamp_aoi(
        min_lon, min_lat, max_lon, max_lat
    )
    return {
        "type": "Polygon",
        "coordinates": [[
            [c_min_lon, c_min_lat],
            [c_max_lon, c_min_lat],
            [c_max_lon, c_max_lat],
            [c_min_lon, c_max_lat],
            [c_min_lon, c_min_lat],
        ]]
    }

In [ ]:
# --------------------------------------------------------------------
# Planet catalog search
# --------------------------------------------------------------------
def search_planet_scenes(session, geometry, incident_date, item_type=PLANET_ITEM_TYPE):
    """Search Planet's catalog for post-event scenes over the AOI.
    Returns a list of scene IDs sorted by cloud cover (ascending).
    """
    after_start = (incident_date + timedelta(days=POST_WINDOW_START)).strftime('%Y-%m-%dT00:00:00.000Z')
    after_end   = (incident_date + timedelta(days=POST_WINDOW_END)).strftime('%Y-%m-%dT23:59:59.999Z')

    query = {
        "item_types": [item_type],
        "filter": {
            "type": "AndFilter",
            "config": [
                {
                    "type": "GeometryFilter",
                    "field_name": "geometry",
                    "config": geometry,
                },
                {
                    "type": "DateRangeFilter",
                    "field_name": "acquired",
                    "config": {
                        "gte": after_start,
                        "lte": after_end,
                    },
                },
                {
                    "type": "RangeFilter",
                    "field_name": "cloud_cover",
                    "config": {
                        "lte": MAX_CLOUD_FRACTION,
                    },
                },
            ],
        },
    }

    try:
        resp = session.post(SEARCH_URL, json=query, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
        data = resp.json()
        features = data.get("features", [])
        parsed = []
        for f in features:
            props = f.get("properties", {})
            cc = props.get("cloud_cover")
            if cc is None:
                continue
            parsed.append({
                "id": f["id"],
                "cloud_cover": cc,
                "acquired": props.get("acquired"),
            })
        parsed.sort(key=lambda x: x["cloud_cover"])
        return parsed
    except Exception as e:
        print(f"    Planet search failed: {e}")
        return []

In [ ]:
# --------------------------------------------------------------------
# Create a Planet order
# --------------------------------------------------------------------
def create_planet_order(session, incident_id, scene_id, geometry):
    """Submit a single Planet order: clip scene to AOI.
    Order name follows the convention: incident_{ID}_after
    """
    order_name = f"incident_{incident_id}_after"

    payload = {
        "name": order_name,
        "products": [
            {
                "item_ids": [scene_id],
                "item_type": PLANET_ITEM_TYPE,
                "product_bundle": PLANET_BUNDLE,
            }
        ],
        "tools": [
            {
                "type": "clip",
                "parameters": {
                    "aoi": geometry,
                },
            }
        ],
    }

    try:
        resp = session.post(ORDERS_URL, json=payload, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
        order = resp.json()
        order_id = order.get("id", "unknown")
        print(f"    Order created: {order_name} (ID: {order_id})")
        return order
    except Exception as e:
        print(f"    Failed to create order for {order_name}: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"    Response: {e.response.text[:500]}")
        return None

In [ ]:
# --------------------------------------------------------------------
# Process a single incident: search → pick best scene → create order
# --------------------------------------------------------------------
def process_incident(row):
    incident_id = int(row['id'])
    incident_date = row['incident_on']
    geometry = aoi_to_planet_geometry(
        row['min_lon'], row['min_lat'], row['max_lon'], row['max_lat']
    )

    print(f"\nIncident {incident_id}: {row['title']}")
    print(f"  Searching Planet catalog for post-event scenes...")

    scenes = search_planet_scenes(session, geometry, incident_date)
    if not scenes:
        print(f"  No suitable post-event scenes found (cloud ≤ {MAX_CLOUD_FRACTION*100:.0f}%).")
        return None

    best = scenes[0]
    print(f"  Best scene: {best['id']} (cloud={best['cloud_cover']:.2f}, date={best['acquired']})")

    order = create_planet_order(session, incident_id, best['id'], geometry)
    if order:
        return {
            "incident_id": incident_id,
            "scene_id": best['id'],
            "order_name": f"incident_{incident_id}_after",
            "order_id": order.get("id"),
            "state": order.get("state"),
        }
    return None

## Run: process all incidents in the index range

For each incident, the notebook searches Planet's catalog for post-event imagery
and submits an order clipped to the clamped AOI. Results are collected in a
summary DataFrame.

In [ ]:
# --------------------------------------------------------------------
# Process incidents (sequential to avoid confusing error messages)
# --------------------------------------------------------------------
results = []
for _, row in df.iterrows():
    result = process_incident(row)
    if result:
        results.append(result)

summary = pd.DataFrame(results)
print(f"\n=== Summary ===")
print(f"Processed {len(df)} incidents.")
print(f"Orders created: {len(results)}")
if not summary.empty:
    print(summary.to_string(index=False))

## Save order summary (for cross-reference)

Saves a CSV with the created order IDs and incident IDs so you can cross-check
with `planet_orders_download.ipynb` later.

In [ ]:
# --------------------------------------------------------------------
# Save order summary to CSV
# --------------------------------------------------------------------
summary_path = "/kaggle/working/planet_orders_summary.csv"
if not summary.empty:
    summary.to_csv(summary_path, index=False)
    print(f"Summary saved to {summary_path}")
    print(f"Columns: {list(summary.columns)}")
else:
    print("No orders were created; nothing to save.")

## Next steps

1. Wait for orders to be fulfilled (typically minutes to hours depending on
   Planet's processing queue). You can check order status at:
   `https://api.planet.com/compute/ops/orders/v2`

2. Once orders are `success`, run `planet_orders_download.ipynb` to download
   the clipped GeoTIFFs.

**Sync tip:** Set the same `start_idx` / `end_idx` in `planet_orders_download.ipynb`
to download only the orders created in this run. The download notebook will filter
by order name (`incident_<ID>_after`) matching the incident IDs in that CSV range.